In [ ]:
# biblioteca para acessar o dataset da UCI
!pip install ucimlrepo

# importando as bibliotecas
import pandas as pd
import numpy as np
from ucimlrepo import fetch_ucirepo

# carregando o dataset
bank_marketing = fetch_ucirepo(id=222)

# separando features (X) e target (y)
X = bank_marketing.data.features
y = bank_marketing.data.targets

# juntando tudo em um único DataFrame para facilitar a análise
df = pd.concat([X, y], axis=1)

# visualizando as primeiras linhas
df.head()

(**OBS**: Os dados foram carregados diretamente do repositório UCI Machine Learning (dataset ID 222 - Bank Marketing), utilizando a biblioteca `ucimlrepo`. Isso garante que o notebook seja executável em qualquer ambiente, sem necessidade de download manual de arquivos.)

## Seção 5.1: Identificação e descrição do problema

**Título do Projeto:**
Previsão de Subscrição de Depósito a Prazo em Campanhas de Marketing Bancário

**Definição da Tarefa:**
- **Atributo-alvo:** A variável `y`, que indica se o cliente subscreveu (1 = "yes") ou não (0 = "no") um depósito a prazo.
- **Atributos Preditivos:** Todas as demais variáveis do dataset (idade, profissão, saldo, tipo de contato, etc.).
- **Tipo de problema:** Classificação Binária, pois o objetivo é prever uma categoria com duas classes possíveis.
- **Objetivo da aplicação:** Construir um modelo que auxilie a instituição bancária a identificar clientes com maior propensão a adquirir o produto, otimizando assim as campanhas de marketing.

**Justificativa:** Este é um problema clássico de marketing direto, onde prever o comportamento do cliente pode reduzir custos e aumentar a eficiência das campanhas telefônicas.

## Seção 5.2: Compreensão dos dados

In [ ]:
# 1. Quantidade de clientes (registros) e colunas (atributos)
print(f"O dataset possui {df.shape[0]} clientes e {df.shape[1]} colunas.")

# 2. Tipos das variáveis e visão geral
print("\n===== Tipos de dados por coluna =====\n")
print(df.dtypes)

print("\n=============== Resumo estatístico das variáveis numéricas ===============\n")
display(df.describe())

print("\n===== Informações gerais do DataFrame =====\n")
df.info()

O dataset possui 45.211 registros e 17 colunas e as variáveis são uma mistura de tipos numéricos (int64) e categóricos (object).

As variáveis numéricas como `age` e `balance` apresentam médias e desvios padrão que sugerem o nível de suas dispersões: A variável `age` apresenta baixa dispersão (std = 10,62), com clientes concentrados entre 30 e 52 anos; Já `balance` mostra alta dispersão (std = 3.044,77) e presença de outliers com saldos muito elevados, sugerindo a necessidade de transformação logarítmica ou tratamento de valores extremos.

A variável `duration` (duração da chamada) apresenta alta variabilidade (std = 257,5), com chamadas de 0 a 4.918 segundos. Embora seja um forte preditor, ela pode causar vazamento de dados (data leakage), pois a duração só é conhecida após o contato. Isso será discutido no pré-processamento.

As variáveis categóricas incluem `job`, `marital`, `education`, etc., que precisarão ser codificadas numericamente para que os algoritmos de Machine Learning possam entendê-las e consigam trabalhar.

In [ ]:
# 3. Verificar valores ausentes
print("=== Valores ausentes por coluna ===\n")
print(df.isnull().sum())

# 4. Verificar duplicatas
print(f"\nNúmero de linhas duplicadas: {df.duplicated().sum()}")

Contrariando a informação inicial da documentação, a análise revelou valores ausentes em algumas variáveis. As colunas `job` (288 ausentes), `education` (1.857 ausentes) e `contact` (13.020 ausentes) apresentam dados faltantes, enquanto `poutcome` possui 36.959 valores ausentes, o que representa 81,7% dos registros. Isso ocorre porque o dataset utiliza *'unknown'* para representar informações não disponíveis, o que é comum em campanhas de marketing. Esses valores deverão ser tratados no pré-processamento.

In [ ]:
# 5. Distribuição da variável alvo
print("\n=== Distribuição da variável 'y' (subscrição) ===\n")
print(df['y'].value_counts())
print(f"\nProporção:\n{df['y'].value_counts(normalize=True) * 100}\n")

# Visualizando em um gráfico de barras
import matplotlib.pyplot as plt
import seaborn as sns

sns.countplot(x='y', data=df)
plt.title('Distribuição da Variável Alvo (y)')
plt.show()

A variável y está desbalanceada. Apenas 11,7% (aproximadamente) dos clientes subscreveram o depósito (yes), enquanto cerca de 88,3% não subscreveram (no). Esse desbalanceamento é comum em campanhas de marketing e deverá ser tratado durante o pré-processamento (por exemplo, com técnicas de balanceamento como SMOTE) ou considerado na escolha das métricas de avaliação (priorizando F1-score em vez de acurácia).

In [ ]:
# Verificando valores únicos em variáveis categóricas para identificar possíveis inconsistências
categorical_cols = df.select_dtypes(include=['object']).columns
for col in categorical_cols:
    print(f"\nValores únicos em '{col}':\n")
    print(df[col].unique())

# Verificar se pdays = -1 realmente significa "não contatado anteriormente"
print("\n\n=== Verificação da variável pdays ===\n")
print(df['pdays'].value_counts().sort_index().head(10))

A variável pdays apresenta um comportamento peculiar: o valor -1, que representa clientes que nunca foram contatados em campanhas anteriores, aparece em 36.954 registros (81,7% do total). Apenas os valores positivos (1 a 9 dias) representam os clientes que já haviam sido contatados. Isso indica que a maioria dos clientes é nova na base de dados, o que é consistente com a natureza de campanhas de marketing. Esse padrão deve ser considerado no pré-processamento, pois -1 não é um valor numérico comum, mas sim uma categoria especial.